In [1]:
#Install packages

!pip install -q pinecone langchain langchain-openai langchain-pinecone langchain-community openai requests beautifulsoup4 plotly


# Import Libraries

import numpy as np
import pandas as pd
import pinecone
import langchain
import openai
import plotly
import bs4
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain.tools import tool

In [8]:
import warnings
warnings.filterwarnings("ignore")

In [16]:
# Mount Google Drive

import os
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# Set your Drive folder path

DRIVE_PATH = "/content/drive/MyDrive/worldcup_data"

PATHS = {
    "results":      Path(DRIVE_PATH) / "results.csv",
    "goalscorers":  Path(DRIVE_PATH) / "goalscorers.csv",
    "shootouts":    Path(DRIVE_PATH) / "shootouts.csv",
    "former_names": Path(DRIVE_PATH) / "former_names.csv",
}


# Verify all  files exist

print("📁 Verifying Google Drive files...\n")
all_found = True
for name, path in PATHS.items():
    if path.exists():
        size = path.stat().st_size / 1024
        print(f"  ✅ {name:<15} {size:.1f} KB")
    else:
        print(f"  ❌ {name:<15} NOT FOUND → {path}")
        all_found = False

if not all_found:
    raise FileNotFoundError(
        f"\n❌ Files missing from Google Drive."
        f"\nExpected folder: {DRIVE_PATH}"
        f"\nMake sure Drive is mounted and folder name matches."
    )

print(f"\n✅ All files found — PATHS variable set")

Mounted at /content/drive
📁 Verifying Google Drive files...

  ✅ results         3611.3 KB
  ✅ goalscorers     3177.2 KB
  ✅ shootouts       28.1 KB
  ✅ former_names    1.7 KB

✅ All files found — PATHS variable set


In [6]:
# Set up API keys

from google.colab import userdata

OPENAI_API_KEY   = userdata.get("OPENAI_API_KEY")
PINECONE_API_KEY = userdata.get("PINECONE_API_KEY")

os.environ["OPENAI_API_KEY"]   = OPENAI_API_KEY
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

## Data Preprocessing

In [17]:
# Step 1: Build name normalization map

def build_name_map(path) -> dict:
    """
    Reads former_names.csv and builds a lookup dictionary.
    Maps old team names → current team names.

    Examples:
      "West Germany"   → "Germany"
      "Soviet Union"   → "Russia"
      "Czechoslovakia" → "Czech Republic"
    """
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()

    name_map = {}
    for _, row in df.iterrows():
        current = str(row["current"]).strip()
        former  = str(row["former"]).strip()
        if former and current and former != current:
            name_map[former] = current

    return name_map


def normalize(name: str, name_map: dict) -> str:
    """Returns current team name — keeps original if not in map."""
    return name_map.get(str(name).strip(), str(name).strip())


name_map = build_name_map(PATHS["former_names"])

print(f"✅ Name map built — {len(name_map)} mappings\n")
print("Sample mappings:")
for k, v in list(name_map.items())[:10]:
    print(f"  '{k}'  →  '{v}'")


✅ Name map built — 36 mappings

Sample mappings:
  'Dahomey'  →  'Benin'
  'Upper Volta'  →  'Burkina Faso'
  'Netherlands Antilles'  →  'Curaçao'
  'Bohemia'  →  'Czechoslovakia'
  'Bohemia and Moravia'  →  'Czechoslovakia'
  'Representation of Czechs and Slovaks'  →  'Czechoslovakia'
  'Belgian Congo'  →  'DR Congo'
  'Congo-Léopoldville'  →  'DR Congo'
  'Congo-Kinshasa'  →  'DR Congo'
  'Zaïre'  →  'DR Congo'


In [18]:
# Load results.csv

def load_results(path, name_map) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()

    df["home_team"] = df["home_team"].apply(lambda x: normalize(x, name_map))
    df["away_team"] = df["away_team"].apply(lambda x: normalize(x, name_map))
    df["date"]      = pd.to_datetime(df["date"])
    df["year"]      = df["date"].dt.year.astype(int)

    df["home_score"] = pd.to_numeric(df["home_score"], errors="coerce").fillna(0).astype(int)
    df["away_score"] = pd.to_numeric(df["away_score"], errors="coerce").fillna(0).astype(int)

    df["tournament"] = df["tournament"].str.strip()
    df["city"]       = df["city"].fillna("").str.strip()
    df["country"]    = df["country"].fillna("").str.strip()

    df = df.dropna(subset=["date","home_team","away_team"]).reset_index(drop=True)
    df = df.rename(columns={"home_score": "home_goals", "away_score": "away_goals"})

    df["result"] = df.apply(
        lambda r: r["home_team"] if r["home_goals"] > r["away_goals"]
        else (r["away_team"] if r["away_goals"] > r["home_goals"] else "draw"),
        axis=1
    )
    df["is_wc"] = df["tournament"].str.contains("FIFA World Cup", case=False, na=False)
    return df

df_all = load_results(PATHS["results"], name_map)
df_wc  = df_all[df_all["is_wc"]].copy().reset_index(drop=True)

print(f"✅ results.csv loaded\n")
print(f"  Total matches:     {len(df_all):,}")
print(f"  World Cup matches: {len(df_wc):,}")
print(f"  WC years:          {sorted(df_wc['year'].unique())}")
print(f"\nSample rows:")
display(df_wc[["date","year","tournament","home_team","away_team","home_goals","away_goals","result"]].head(3))


✅ results.csv loaded

  Total matches:     49,071
  World Cup matches: 9,719
  WC years:          [1930, 1933, 1934, 1937, 1938, 1949, 1950, 1953, 1954, 1956, 1957, 1958, 1960, 1961, 1962, 1964, 1965, 1966, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1976, 1977, 1978, 1980, 1981, 1982, 1984, 1985, 1986, 1988, 1989, 1990, 1992, 1993, 1994, 1996, 1997, 1998, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Sample rows:


,date,year,tournament,home_team,away_team,home_goals,away_goals,result
0,1930-07-13,1930,FIFA World Cup,Belgium,United States,0,3,United States
1,1930-07-13,1930,FIFA World Cup,France,Mexico,4,1,France
2,1930-07-14,1930,FIFA World Cup,Brazil,Yugoslavia,1,2,Yugoslavia


In [19]:
# Load goalscorers.csv

def load_goalscorers(path, name_map) -> dict:
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()

    df["home_team"] = df["home_team"].apply(lambda x: normalize(x, name_map))
    df["away_team"] = df["away_team"].apply(lambda x: normalize(x, name_map))
    df["team"]      = df["team"].apply(lambda x: normalize(x, name_map))
    df["date"]      = pd.to_datetime(df["date"])
    df["scorer"]    = df["scorer"].fillna("Unknown").str.strip()
    df["own_goal"]  = df["own_goal"].fillna(False).astype(bool)
    df["penalty"]   = df["penalty"].fillna(False).astype(bool)

    scorer_lookup = {}
    for (date, home, away), group in df.groupby(["date","home_team","away_team"]):
        key     = (str(date.date()), home, away)
        scorers = {"home": [], "away": []}
        for _, row in group.iterrows():
            note       = " (OG)" if row["own_goal"] else (" (pen)" if row["penalty"] else "")
            scorer_str = f"{row['scorer']}{note}"
            if row["team"] == home:
                scorers["home"].append(scorer_str)
            else:
                scorers["away"].append(scorer_str)
        scorer_lookup[key] = scorers

    return scorer_lookup

scorer_lookup = load_goalscorers(PATHS["goalscorers"], name_map)

print(f"✅ goalscorers.csv loaded — {len(scorer_lookup):,} matches have scorer data\n")
sample_key = list(scorer_lookup.keys())[500]
print(f"Sample entry:")
print(f"  Match:   {sample_key}")
print(f"  Scorers: {scorer_lookup[sample_key]}")


✅ goalscorers.csv loaded — 15,425 matches have scorer data

Sample entry:
  Match:   ('1953-11-11', 'England', 'Northern Ireland')
  Scorers: {'home': ['Harold Hassall', 'Harold Hassall', 'Nat Lofthouse'], 'away': ['Eddie McMorran']}


In [20]:
# Load shootouts.csv

def load_shootouts(path, name_map) -> dict:
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()

    df["home_team"] = df["home_team"].apply(lambda x: normalize(x, name_map))
    df["away_team"] = df["away_team"].apply(lambda x: normalize(x, name_map))
    df["winner"]    = df["winner"].apply(lambda x: normalize(x, name_map))
    df["date"]      = pd.to_datetime(df["date"])

    shootout_lookup = {}
    for _, row in df.iterrows():
        key = (str(row["date"].date()), row["home_team"], row["away_team"])
        shootout_lookup[key] = row["winner"]

    return shootout_lookup

shootout_lookup = load_shootouts(PATHS["shootouts"], name_map)

print(f"✅ shootouts.csv loaded — {len(shootout_lookup):,} shootout records\n")
sample_key = list(shootout_lookup.keys())[0]
print(f"Sample entry:")
print(f"  Match:  {sample_key}")
print(f"  Winner: {shootout_lookup[sample_key]}")


✅ shootouts.csv loaded — 665 shootout records

Sample entry:
  Match:  ('1967-08-22', 'India', 'Taiwan')
  Winner: Taiwan


In [23]:
# Compute team stats


def compute_team_stats(df_wc, scorer_lookup, shootout_lookup) -> pd.DataFrame:
    STAGE_RANK = {
        "final": 8, "third place": 7,
        "semi-final": 6, "semi-finals": 6,
        "quarter-final": 5, "quarter-finals": 5,
        "round of 16": 4, "second round": 4,
        "group": 3, "first round": 2,
    }

    def rank_stage(stage):
        s = str(stage).lower()
        for k, v in STAGE_RANK.items():
            if k in s: return v
        return 0

    all_teams = sorted(set(df_wc["home_team"]) | set(df_wc["away_team"]))
    stats     = []

    for team in all_teams:
        hm = df_wc["home_team"] == team
        am = df_wc["away_team"] == team
        g  = df_wc[hm | am].copy()
        if len(g) == 0:
            continue

        played = len(g)
        wins   = sum(g["result"] == team)
        draws  = sum(g["result"] == "draw")
        losses = played - wins - draws

        goals_scored   = g[hm]["home_goals"].sum() + g[am]["away_goals"].sum()
        goals_conceded = g[hm]["away_goals"].sum() + g[am]["home_goals"].sum()

        best_rank  = g["tournament"].apply(rank_stage).max()
        best_stage = next(
            (k.title() for k, v in STAGE_RANK.items() if v == best_rank),
            "Group Stage"
        )

        appearances = len(g["year"].unique())
        tournaments = sorted(g["year"].unique().tolist())

        # Top scorers
        team_scorers = {}
        for _, row in g.iterrows():
            key  = (str(row["date"].date()), row["home_team"], row["away_team"])
            side = "home" if row["home_team"] == team else "away"
            if key in scorer_lookup:
                for s in scorer_lookup[key].get(side, []):
                    base = s.split(" (")[0]
                    team_scorers[base] = team_scorers.get(base, 0) + 1

        top_5 = sorted(team_scorers.items(), key=lambda x: x[1], reverse=True)[:5]
        top_scorers_str = ", ".join(f"{n} ({g})" for n, g in top_5) if top_5 else "No data"

        # Penalty record
        pen_wins = pen_losses = 0
        for _, row in g.iterrows():
            key = (str(row["date"].date()), row["home_team"], row["away_team"])
            if key in shootout_lookup:
                if shootout_lookup[key] == team: pen_wins += 1
                else: pen_losses += 1

        stats.append({
            "team":               team,
            "appearances":        appearances,
            "tournaments":        str(tournaments),
            "matches_played":     played,
            "wins":               wins,
            "draws":              draws,
            "losses":             losses,
            "win_rate":           round(wins / played * 100, 1),
            "goals_scored":       int(goals_scored),
            "goals_conceded":     int(goals_conceded),
            "avg_goals_scored":   round(goals_scored / played, 2),
            "avg_goals_conceded": round(goals_conceded / played, 2),
            "best_stage":         best_stage,
            "top_scorers":        top_scorers_str,
            "penalty_wins":       pen_wins,
            "penalty_losses":     pen_losses,
        })

    return pd.DataFrame(stats).sort_values("wins", ascending=False).reset_index(drop=True)


df_stats = compute_team_stats(df_wc, scorer_lookup, shootout_lookup)

print(f"✅ Team stats computed — {len(df_stats)} teams\n")
print("🏆 Top 10 by World Cup wins:")
display(df_stats[[
    "team","appearances","wins","win_rate","goals_scored","best_stage","top_scorers"
]].head(10))

✅ Team stats computed — 217 teams

🏆 Top 10 by World Cup wins:


,team,appearances,wins,win_rate,goals_scored,best_stage,top_scorers
0,Brazil,45,166,64.1,540,Group Stage,"Ronaldo (25), Neymar (24), Pelé (18), Rivaldo ..."
1,Germany,46,156,70.3,576,Group Stage,"Miroslav Klose (29), Gerd Müller (23), Karl-He..."
2,Argentina,46,145,56.0,445,Group Stage,"Lionel Messi (49), Hernán Crespo (23), Gabriel..."
3,Italy,49,129,61.7,383,Group Stage,"Roberto Baggio (15), Roberto Bettega (13), Chr..."
4,Netherlands,44,125,63.1,452,Group Stage,"Memphis Depay (26), Robin van Persie (19), Den..."
5,Spain,47,123,62.1,420,Group Stage,"David Villa (17), Fernando Hierro (15), Raúl (..."
6,Mexico,40,122,55.5,452,Group Stage,"Jared Borgetti (25), Carlos Hermosillo (15), C..."
7,England,47,121,60.2,429,Group Stage,"Harry Kane (33), Wayne Rooney (17), Gary Linek..."
8,France,48,114,57.6,386,Group Stage,"Kylian Mbappé (23), Michel Platini (17), Just ..."
9,South Korea,36,111,54.1,375,Group Stage,"Son Heung-min (28), Park Chu-young (13), Ha Se..."


In [24]:
# Sanity checks

print("🔍 Running sanity checks...\n")

# Check 1 — Name normalization worked
germany_variants = set(
    df_wc[df_wc["home_team"].str.contains("Germany", case=False)]["home_team"].unique()
) | set(
    df_wc[df_wc["away_team"].str.contains("Germany", case=False)]["away_team"].unique()
)
status = "✅" if germany_variants == {"Germany"} else "⚠️"
print(f"{status} Germany name variants: {germany_variants}")
print(f"   Expected: {{'Germany'}} only\n")

# Check 2 — 2022 World Cup present
wc_2022 = df_wc[df_wc["year"] == 2022]
status  = "✅" if len(wc_2022) >= 60 else "⚠️"
print(f"{status} 2022 World Cup matches: {len(wc_2022)}")
print(f"   Expected: ~64 matches\n")

# Check 3 — Germany vs Brazil 2014 scorer data
found_2014 = False
for _, row in df_wc[df_wc["year"] == 2014].iterrows():
    teams = {row["home_team"], row["away_team"]}
    if "Germany" in teams and "Brazil" in teams:
        key = (str(row["date"].date()), row["home_team"], row["away_team"])
        if key in scorer_lookup:
            print(f"✅ Germany vs Brazil 2014 scorers:")
            print(f"   {scorer_lookup[key]}\n")
            found_2014 = True
        break
if not found_2014:
    print("⚠️  Germany vs Brazil 2014 scorer data not found\n")

# Check 4 — Shootout records
print(f"✅ Total shootout records: {len(shootout_lookup)}")
print(f"   Expected: 30+ records\n")

# Check 5 — Brazil stats
brazil = df_stats[df_stats["team"] == "Brazil"]
if len(brazil) > 0:
    b = brazil.iloc[0]
    status = "✅" if b["appearances"] >= 20 else "⚠️"
    print(f"{status} Brazil — Appearances: {b['appearances']} | "
          f"Wins: {b['wins']} | Win rate: {b['win_rate']}%")
    print(f"   Top scorers: {b['top_scorers']}\n")

print("─" * 50)
print("✅ Sanity checks done")
print("\nVariables ready for next step:")
print("  df_all         → all 49,016 international matches")
print("  df_wc          → World Cup matches only")
print("  df_stats       → team statistics")
print("  scorer_lookup  → match → scorers dictionary")
print("  shootout_lookup→ match → pen winner dictionary")
print("\nNext → convert to NL documents + store in Pinecone")

🔍 Running sanity checks...

✅ Germany name variants: {'Germany'}
   Expected: {'Germany'} only

✅ 2022 World Cup matches: 164
   Expected: ~64 matches

✅ Germany vs Brazil 2014 scorers:
   {'home': ['Oscar'], 'away': ['Thomas Müller', 'Miroslav Klose', 'Toni Kroos', 'Toni Kroos', 'Sami Khedira', 'André Schürrle', 'André Schürrle']}

✅ Total shootout records: 665
   Expected: 30+ records

✅ Brazil — Appearances: 45 | Wins: 166 | Win rate: 64.1%
   Top scorers: Ronaldo (25), Neymar (24), Pelé (18), Rivaldo (17), Zico (16)

──────────────────────────────────────────────────
✅ Sanity checks done

Variables ready for next step:
  df_all         → all 49,016 international matches
  df_wc          → World Cup matches only
  df_stats       → team statistics
  scorer_lookup  → match → scorers dictionary
  shootout_lookup→ match → pen winner dictionary

Next → convert to NL documents + store in Pinecone


In [25]:
# Inspect tournament names to fix filter

import pandas as pd

# See ALL tournament names containing "World Cup"
wc_tournaments = df_all[
    df_all["tournament"].str.contains("FIFA World Cup", case=False, na=False)
]["tournament"].value_counts()

print("📋 All FIFA World Cup tournament name variants:\n")
print(wc_tournaments.to_string())


📋 All FIFA World Cup tournament name variants:

tournament
FIFA World Cup qualification    8755
FIFA World Cup                   964


In [26]:
# Fix the filter: tournament matches only
# ============================================================
# Typically the actual tournament is just "FIFA World Cup"
# Qualifiers are "FIFA World Cup qualification" or similar

# Step 1 — See exactly what names exist
wc_names = df_all[
    df_all["tournament"].str.contains("FIFA World Cup", case=False, na=False)
]["tournament"].unique()

print("All WC-related tournament names:")
for name in sorted(wc_names):
    count = len(df_all[df_all["tournament"] == name])
    print(f"  '{name}'  →  {count:,} matches")

print()

# Step 2 — Filter to EXACT tournament match only
# Excludes: qualification, qualifying, qualifier
EXACT_WC_NAME = "FIFA World Cup"

df_wc = df_all[
    df_all["tournament"] == EXACT_WC_NAME
].copy().reset_index(drop=True)

print(f"\n✅ Fixed filter: tournament == '{EXACT_WC_NAME}' exactly")
print(f"   World Cup matches now: {len(df_wc):,}")
print(f"   WC years: {sorted(df_wc['year'].unique())}")



All WC-related tournament names:
  'FIFA World Cup'  →  964 matches
  'FIFA World Cup qualification'  →  8,755 matches


✅ Fixed filter: tournament == 'FIFA World Cup' exactly
   World Cup matches now: 964
   WC years: [1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970, 1974, 1978, 1982, 1986, 1990, 1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022]


In [27]:
# Recompute team stats with fixed df_wc

df_stats = compute_team_stats(df_wc, scorer_lookup, shootout_lookup)

print(f"\n✅ Team stats recomputed — {len(df_stats)} teams\n")
print("🏆 Top 10 by World Cup wins:")
display(df_stats[[
    "team","appearances","wins","win_rate","goals_scored","best_stage","top_scorers"
]].head(10))




✅ Team stats recomputed — 82 teams

🏆 Top 10 by World Cup wins:


,team,appearances,wins,win_rate,goals_scored,best_stage,top_scorers
0,Brazil,22,76,66.7,237,Group Stage,"Ronaldo (15), Pelé (12), Ademir de Menezes (9)..."
1,Germany,20,68,60.7,232,Group Stage,"Miroslav Klose (16), Gerd Müller (14), Jürgen ..."
2,Argentina,18,47,53.4,152,Group Stage,"Lionel Messi (13), Gabriel Batistuta (10), Gui..."
3,Italy,18,45,54.2,128,Group Stage,"Paolo Rossi (9), Roberto Baggio (9), Christian..."
4,France,16,39,53.4,136,Group Stage,"Just Fontaine (13), Kylian Mbappé (12), Thierr..."
5,England,16,32,43.2,104,Group Stage,"Gary Lineker (10), Harry Kane (8), Geoff Hurst..."
6,Spain,16,31,46.3,108,Group Stage,"David Villa (9), Emilio Butragueño (5), Fernan..."
7,Netherlands,11,30,54.5,96,Group Stage,"Johnny Rep (7), Rob Rensenbrink (6), Dennis Be..."
8,Uruguay,14,25,42.4,89,Group Stage,"Óscar Míguez (8), Luis Suárez (7), Diego Forlá..."
9,Belgium,14,21,41.2,69,Group Stage,"Marc Wilmots (5), Romelu Lukaku (5), Jan Ceule..."


In [28]:
# Re-run sanity checks with fixed data

print("🔍 Re-running sanity checks...\n")

# Check 1 — 2022 match count
wc_2022 = df_wc[df_wc["year"] == 2022]
status  = "✅" if 60 <= len(wc_2022) <= 70 else "⚠️"
print(f"{status} 2022 World Cup matches: {len(wc_2022)}")
print(f"   Expected: ~64 matches\n")

# Check 2 — Brazil appearances
brazil = df_stats[df_stats["team"] == "Brazil"].iloc[0]
status = "✅" if 20 <= brazil["appearances"] <= 25 else "⚠️"
print(f"{status} Brazil appearances: {brazil['appearances']}")
print(f"   Expected: ~22\n")

# Check 3 — Brazil wins
status = "✅" if 60 <= brazil["wins"] <= 80 else "⚠️"
print(f"{status} Brazil wins: {brazil['wins']}")
print(f"   Expected: ~70\n")

# Check 4 — Germany name still clean
germany_variants = set(
    df_wc[df_wc["home_team"].str.contains("Germany", case=False)]["home_team"].unique()
) | set(
    df_wc[df_wc["away_team"].str.contains("Germany", case=False)]["away_team"].unique()
)
status = "✅" if germany_variants == {"Germany"} else "⚠️"
print(f"{status} Germany name variants: {germany_variants}\n")

# Check 5 — Total WC matches reasonable
status = "✅" if 700 <= len(df_wc) <= 1000 else "⚠️"
print(f"{status} Total WC matches: {len(df_wc):,}")
print(f"   Expected: 700–900 range\n")

# Check 6 — Matches per year sample
print("📊 Matches per year (recent tournaments):")
recent = df_wc[df_wc["year"] >= 2010].groupby("year").size()
print(recent.to_string())
print(f"\n   Expected: ~64 per year (2026 onwards will have 104)\n")

print("─" * 50)

all_good = (
    60 <= len(wc_2022) <= 70 and
    20 <= brazil["appearances"] <= 25 and
    60 <= brazil["wins"] <= 80 and
    germany_variants == {"Germany"} and
    700 <= len(df_wc) <= 1000
)

if all_good:
    print("✅ All checks passed — data is clean")
    print("   Ready to convert to NL documents")
else:
    print("⚠️  Some checks failed — review output above")
    print("   Check Cell H for exact tournament names")

🔍 Re-running sanity checks...

✅ 2022 World Cup matches: 64
   Expected: ~64 matches

✅ Brazil appearances: 22
   Expected: ~22

✅ Brazil wins: 76
   Expected: ~70

✅ Germany name variants: {'Germany'}

✅ Total WC matches: 964
   Expected: 700–900 range

📊 Matches per year (recent tournaments):
year
2010    64
2014    64
2018    64
2022    64

   Expected: ~64 per year (2026 onwards will have 104)

──────────────────────────────────────────────────
✅ All checks passed — data is clean
   Ready to convert to NL documents


## Embedding data to Vector Database

In [30]:
# Convert matches to NL documents

from langchain_core.documents import Document

def match_to_text(row, scorer_lookup, shootout_lookup) -> str:
    """
    Converts one match row to a rich natural language sentence.
    Includes: teams, score, result, scorers, shootout if applicable.
    """
    hg, ag     = int(row["home_goals"]), int(row["away_goals"])
    home, away = str(row["home_team"]),  str(row["away_team"])
    year       = int(row["year"])
    tournament = str(row.get("tournament", "international match"))
    city       = str(row.get("city", ""))

    # Result phrase
    if hg > ag:
        margin = hg - ag
        dom    = (
            "in a dominant performance" if margin >= 5 else
            "comfortably"              if margin >= 3 else
            "narrowly"
        )
        result_phrase = f"{home} defeated {away} {hg}–{ag} {dom}"
        pen_text      = ""

    elif ag > hg:
        margin = ag - hg
        dom    = (
            "in a dominant performance" if margin >= 5 else
            "comfortably"              if margin >= 3 else
            "narrowly"
        )
        result_phrase = f"{away} defeated {home} {ag}–{hg} {dom}"
        pen_text      = ""

    else:
        result_phrase = f"{home} and {away} drew {hg}–{ag}"
        # Check for penalty shootout winner
        key = (str(row["date"].date()), home, away)
        if key in shootout_lookup:
            winner   = shootout_lookup[key]
            pen_text = f" {winner} won on penalty shootout."
        else:
            pen_text = ""

    # High scoring note
    total      = hg + ag
    goals_note = (
        f" It was a high-scoring game with {total} total goals."
        if total >= 6 else ""
    )

    # Scorer details
    key = (str(row["date"].date()), home, away)
    if key in scorer_lookup:
        sc       = scorer_lookup[key]
        home_sc  = ", ".join(sc["home"]) if sc["home"] else "none"
        away_sc  = ", ".join(sc["away"]) if sc["away"] else "none"
        sc_text  = f" Scorers — {home}: {home_sc}. {away}: {away_sc}."
    else:
        sc_text  = ""

    # Location
    loc = f" in {city}," if city and city != "nan" else ""

    return (
        f"In the {year} {tournament}{loc} "
        f"{result_phrase}.{goals_note}{sc_text}{pen_text}"
    ).strip()


def matches_to_docs(df, scorer_lookup, shootout_lookup) -> list:
    """Converts match DataFrame to list of LangChain Documents."""
    docs = []
    for _, r in df.iterrows():
        hg, ag     = int(r["home_goals"]), int(r["away_goals"])
        home, away = str(r["home_team"]),  str(r["away_team"])
        total      = hg + ag

        # Shootout info
        key          = (str(r["date"].date()), home, away)
        had_shootout = key in shootout_lookup
        pen_winner   = shootout_lookup.get(key, "")

        # Scorer lists for metadata
        home_sc = []
        away_sc = []
        if key in scorer_lookup:
            home_sc = [s.split(" (")[0] for s in scorer_lookup[key]["home"]]
            away_sc = [s.split(" (")[0] for s in scorer_lookup[key]["away"]]

        docs.append(Document(
            page_content=match_to_text(r, scorer_lookup, shootout_lookup),
            metadata={
                "date":         str(r["date"].date()),
                "year":         int(r["year"]),
                "tournament":   str(r.get("tournament", "")),
                "home_team":    home,
                "away_team":    away,
                "home_goals":   hg,
                "away_goals":   ag,
                "winner":       (
                    home       if hg > ag else
                    away       if ag > hg else
                    pen_winner if pen_winner else "draw"
                ),
                "total_goals":  total,
                "is_wc":        bool(r.get("is_wc", False)),
                "is_final":     "final" in str(r.get("tournament","")).lower(),
                "had_shootout": had_shootout,
                "pen_winner":   pen_winner,
                "city":         str(r.get("city", "")),
            }
        ))
    return docs


# Build WC match documents
print("⚙️  Converting WC matches to documents...")
wc_match_docs = matches_to_docs(df_wc, scorer_lookup, shootout_lookup)
print(f"✅ WC match docs:  {len(wc_match_docs):,}")

# Build international match documents
print("⚙️  Converting all international matches to documents...")
intl_match_docs = matches_to_docs(df_all, scorer_lookup, shootout_lookup)
print(f"✅ Intl match docs: {len(intl_match_docs):,}")

# Preview sample documents
print(f"\n📄 Sample WC match document:")
print(f"   {wc_match_docs[200].page_content}")
print(f"\n📄 Sample intl document:")
print(f"   {intl_match_docs[1000].page_content}")


⚙️  Converting WC matches to documents...
✅ WC match docs:  964
⚙️  Converting all international matches to documents...
✅ Intl match docs: 49,071

📄 Sample WC match document:
   In the 1970 FIFA World Cup in Mexico City, Mexico and Russia drew 0–0.

📄 Sample intl document:
   In the 1925 Friendly in Zürich, Switzerland defeated Netherlands 4–1 comfortably.


In [31]:
# Convert team stats to NL documents
# One document per team — includes top scorers + pen record

def stats_to_docs(df_stats) -> list:
    docs = []
    for _, r in df_stats.iterrows():
        # Penalty record sentence
        pen_total = r["penalty_wins"] + r["penalty_losses"]
        if pen_total > 0:
            pen_text = (
                f"In penalty shootouts they have won {r['penalty_wins']} "
                f"and lost {r['penalty_losses']}."
            )
        else:
            pen_text = "They have not appeared in a penalty shootout."

        text = (
            f"{r['team']} have participated in {r['appearances']} FIFA World Cups "
            f"({r['tournaments']}), playing {r['matches_played']} matches. "
            f"Their record is {r['wins']} wins, {r['draws']} draws, "
            f"{r['losses']} losses — a win rate of {r['win_rate']}%. "
            f"They have scored {r['goals_scored']} goals and conceded "
            f"{r['goals_conceded']}, averaging {r['avg_goals_scored']} scored "
            f"and {r['avg_goals_conceded']} conceded per game. "
            f"Best stage reached: {r['best_stage']}. "
            f"All-time top World Cup scorers: {r['top_scorers']}. "
            f"{pen_text}"
        )

        docs.append(Document(
            page_content=text,
            metadata={
                "team":               str(r["team"]),
                "appearances":        int(r["appearances"]),
                "wins":               int(r["wins"]),
                "draws":              int(r["draws"]),
                "losses":             int(r["losses"]),
                "win_rate":           float(r["win_rate"]),
                "goals_scored":       int(r["goals_scored"]),
                "goals_conceded":     int(r["goals_conceded"]),
                "avg_goals_scored":   float(r["avg_goals_scored"]),
                "avg_goals_conceded": float(r["avg_goals_conceded"]),
                "best_stage":         str(r["best_stage"]),
                "penalty_wins":       int(r["penalty_wins"]),
                "penalty_losses":     int(r["penalty_losses"]),
            }
        ))
    return docs


stats_docs = stats_to_docs(df_stats)
print(f"✅ Stats docs: {len(stats_docs):,}")
print(f"\n📄 Sample stats document:")
print(f"   {stats_docs[0].page_content}")


✅ Stats docs: 82

📄 Sample stats document:
   Brazil have participated in 22 FIFA World Cups ([1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970, 1974, 1978, 1982, 1986, 1990, 1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022]), playing 114 matches. Their record is 76 wins, 19 draws, 19 losses — a win rate of 66.7%. They have scored 237 goals and conceded 108, averaging 2.08 scored and 0.95 conceded per game. Best stage reached: Group Stage. All-time top World Cup scorers: Ronaldo (15), Pelé (12), Ademir de Menezes (9), Vavá (9), Jairzinho (9). In penalty shootouts they have won 3 and lost 2.


In [32]:
# Scrape Wikipedia 2026 + convert to documents

import time, requests
from pathlib import Path
from bs4 import BeautifulSoup

CACHE_DIR = Path("/content/cache")
CACHE_DIR.mkdir(exist_ok=True)

def scrape_wikipedia_2026() -> dict:
    cache_file = CACHE_DIR / "wikipedia_2026.html"
    url        = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup"

    if cache_file.exists():
        print("📂 Loading from Colab cache")
        html = cache_file.read_text(encoding="utf-8")
    else:
        print(f"🌐 Scraping Wikipedia (2s rate limit)...")
        time.sleep(2)
        try:
            resp = requests.get(
                url,
                headers={"User-Agent": "Mozilla/5.0 (educational research bot)"},
                timeout=15
            )
            resp.raise_for_status()
            html = resp.text
            cache_file.write_text(html, encoding="utf-8")
            print("✅ Scraped and cached")
        except Exception as e:
            print(f"⚠️ Scrape failed: {e} — using fallback")
            return _fallback_2026()

    soup  = BeautifulSoup(html, "html.parser")
    facts = [
        p.get_text(strip=True)[:400]
        for p in soup.find_all("p")[:15]
        if len(p.get_text(strip=True)) > 60
        and "World Cup" in p.get_text()
    ]

    known_teams = [
        "United States","Canada","Mexico","Brazil","Argentina",
        "France","England","Germany","Spain","Portugal",
        "Netherlands","Belgium","Croatia","Morocco","Japan",
        "South Korea","Australia","Ecuador","Uruguay","Colombia",
        "Senegal","Egypt","Saudi Arabia","New Zealand","Serbia",
        "Poland","Switzerland","Denmark","Austria","Turkey",
        "Peru","Nigeria","Ghana","Cameroon","Slovenia","Albania",
        "Hungary","Romania","Ukraine","Venezuela","Panama",
        "Costa Rica","Jamaica","Honduras","IR Iran","Qatar",
    ]
    page_text   = soup.get_text()
    found_teams = [t for t in known_teams if t in page_text]

    return {
        "host_info": (
            "The 2026 FIFA World Cup will be jointly hosted by "
            "the United States, Canada, and Mexico — the first "
            "World Cup hosted by three nations simultaneously. "
            "It is the first edition to feature 48 teams, expanded "
            "from the previous 32-team format. Matches will be played "
            "across 16 host cities. The final will be held at MetLife "
            "Stadium in East Rutherford, New Jersey, USA."
        ),
        "qualified_teams": found_teams,
        "facts":           facts[:10],
    }


def _fallback_2026() -> dict:
    return {
        "host_info": (
            "The 2026 FIFA World Cup is hosted by USA, Canada, and Mexico. "
            "First edition with 48 teams. "
            "Final at MetLife Stadium, New Jersey."
        ),
        "qualified_teams": [
            "United States","Canada","Mexico","Brazil","Argentina",
            "France","England","Germany","Spain","Portugal",
            "Netherlands","Belgium","Croatia","Morocco","Japan",
            "South Korea","Australia","Ecuador","Uruguay","Colombia",
        ],
        "facts": [
            "The 2026 FIFA World Cup features 48 teams for the first time.",
            "Hosted by the United States, Canada, and Mexico.",
            "The final will be held at MetLife Stadium in New Jersey.",
        ],
    }


def wc2026_to_docs(data) -> list:
    docs = [Document(
        page_content=data["host_info"],
        metadata={"type": "host_info", "year": 2026}
    )]
    for team in data["qualified_teams"]:
        docs.append(Document(
            page_content=(
                f"{team} have qualified for the 2026 FIFA World Cup, "
                f"hosted jointly by the United States, Canada, and Mexico."
            ),
            metadata={"type": "qualified_team", "team": team, "year": 2026}
        ))
    for fact in data["facts"]:
        if len(fact) > 30:
            docs.append(Document(
                page_content=fact,
                metadata={"type": "fact", "year": 2026}
            ))
    return docs


wc_2026      = scrape_wikipedia_2026()
preview_docs = wc2026_to_docs(wc_2026)

print(f"\n✅ 2026 preview docs: {len(preview_docs)}")
print(f"   Teams found: {len(wc_2026['qualified_teams'])}")
print(f"   Facts extracted: {len(wc_2026['facts'])}")



🌐 Scraping Wikipedia (2s rate limit)...
✅ Scraped and cached

✅ 2026 preview docs: 47
   Teams found: 38
   Facts extracted: 8


In [35]:
# Final document summary before embedding

print("📊 Document summary — ready for Pinecone\n")
print(f"{'Index':<30} {'Documents':>10}")
print("─" * 42)
print(f"  {'worldcup-matches':<28} {len(wc_match_docs):>10,}")
print(f"  {'international-matches':<28} {len(intl_match_docs):>10,}")
print(f"  {'worldcup-team-stats':<28} {len(stats_docs):>10,}")
print(f"  {'worldcup-2026':<28} {len(preview_docs):>10,}")
print("─" * 42)
total = len(wc_match_docs)+len(intl_match_docs)+len(stats_docs)+len(preview_docs)
print(f"  {'TOTAL':<28} {total:>10,}")


📊 Document summary — ready for Pinecone

Index                           Documents
──────────────────────────────────────────
  worldcup-matches                    964
  international-matches            49,071
  worldcup-team-stats                  82
  worldcup-2026                        47
──────────────────────────────────────────
  TOTAL                            50,164


In [38]:
# Check Pinecone connection + create missing indexes

import time
from pinecone import Pinecone, ServerlessSpec

# Step 1 — See what indexes currently exist
existing = [idx.name for idx in pc.list_indexes()]
print(f"📋 Indexes currently in your Pinecone account:")
if existing:
    for name in existing:
        info  = pc.describe_index(name)
        count = pc.Index(name).describe_index_stats().get("total_vector_count", 0)
        print(f"  ✅ {name:<35} dim={info.dimension} vectors={count:,}")
else:
    print("  ⚠️  No indexes found — will create all 4 now")

print()

# Step 2 — Define the 4 indexes we need
INDEXES = {
    "wc_matches":   "worldcup-matches",
    "intl_matches": "international-matches",
    "team_stats":   "worldcup-team-stats",
    "wc_2026":      "worldcup-2026",
}

# Step 3 — Create any missing indexes
def create_index_if_needed(name: str):
    existing = [idx.name for idx in pc.list_indexes()]
    if name in existing:
        print(f"  📂 Already exists: {name}")
        return

    print(f"  ⚙️  Creating: {name} ...")
    pc.create_index(
        name=name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    # Wait until ready
    while not pc.describe_index(name).status["ready"]:
        print(f"     ⏳ Waiting for {name}...")
        time.sleep(2)

    print(f"  ✅ Created and ready: {name}")


print("🔧 Creating missing indexes...\n")
for label, name in INDEXES.items():
    create_index_if_needed(name)

# Step 4 — Final verification
print("\n📊 Final index status:\n")
print(f"{'Index':<35} {'Dim':<8} {'Vectors':<10} {'Status'}")
print("─" * 65)
all_ready = True
for label, name in INDEXES.items():
    try:
        info   = pc.describe_index(name)
        count  = pc.Index(name).describe_index_stats().get("total_vector_count", 0)
        status = "✅ Ready" if info.status["ready"] else "⏳ Not ready"
        print(f"  {name:<33} {info.dimension:<8} {count:<10,} {status}")
    except Exception as e:
        print(f"  {name:<33} ❌ Error: {e}")
        all_ready = False

print()
if all_ready:
    print("✅ All indexes ready — proceed to Cell P to start embedding")
else:
    print("❌ Some indexes failed — check error above")
    print("   Common fix: verify PINECONE_API_KEY is correct in Colab Secrets")

📋 Indexes currently in your Pinecone account:
  ⚠️  No indexes found — will create all 4 now

🔧 Creating missing indexes...

  ⚙️  Creating: worldcup-matches ...
  ✅ Created and ready: worldcup-matches
  ⚙️  Creating: international-matches ...
  ✅ Created and ready: international-matches
  ⚙️  Creating: worldcup-team-stats ...
  ✅ Created and ready: worldcup-team-stats
  ⚙️  Creating: worldcup-2026 ...
  ✅ Created and ready: worldcup-2026

📊 Final index status:

Index                               Dim      Vectors    Status
─────────────────────────────────────────────────────────────────
  worldcup-matches                  1536     0          ✅ Ready
  international-matches             1536     0          ✅ Ready
  worldcup-team-stats               1536     0          ✅ Ready
  worldcup-2026                     1536     0          ✅ Ready

✅ All indexes ready — proceed to Cell P to start embedding


In [39]:
# Store all 4 indexes in Pinecone

from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
pc         = Pinecone(api_key=PINECONE_API_KEY)

INDEXES = {
    "wc_matches":   "worldcup-matches",
    "intl_matches": "international-matches",
    "team_stats":   "worldcup-team-stats",
    "wc_2026":      "worldcup-2026",
}

def store_in_pinecone(docs, index_name, batch_size=100):
    """
    Stores documents in Pinecone in batches.
    Skips if index already has data.
    Shows progress + ETA.
    """
    current = pc.Index(index_name).describe_index_stats().get(
        "total_vector_count", 0
    )
    if current > 0:
        print(f"⏭️  '{index_name}' already has {current:,} vectors — skipping")
        return

    total_batches = (len(docs) + batch_size - 1) // batch_size
    print(f"\n📤 Uploading to '{index_name}'")
    print(f"   {len(docs):,} docs → {total_batches} batches\n")

    start = time.time()
    for i in range(0, len(docs), batch_size):
        batch     = docs[i:i+batch_size]
        batch_num = i // batch_size + 1
        done      = min(i + batch_size, len(docs))

        PineconeVectorStore.from_documents(
            documents=batch,
            embedding=embeddings,
            index_name=index_name,
        )

        elapsed = time.time() - start
        pct     = done / len(docs) * 100
        eta     = (elapsed / done) * (len(docs) - done) if done > 0 else 0

        print(
            f"  Batch {batch_num:>4}/{total_batches} | "
            f"{done:>6,}/{len(docs):,} | "
            f"{pct:>5.1f}% | "
            f"ETA {eta/60:.1f} min",
            end="\r"
        )
        time.sleep(0.3)

    print(f"\n  ✅ Done — {len(docs):,} docs in '{index_name}'")


print("🚀 Starting Pinecone ingestion...\n")
print("=" * 55)

# Store in recommended order — smallest first
store_in_pinecone(stats_docs,      INDEXES["team_stats"])
store_in_pinecone(preview_docs,    INDEXES["wc_2026"])
store_in_pinecone(wc_match_docs,   INDEXES["wc_matches"])
store_in_pinecone(intl_match_docs, INDEXES["intl_matches"])  # longest last

print("\n" + "=" * 55)
print("✅ All 4 indexes populated in Pinecone")


🚀 Starting Pinecone ingestion...


📤 Uploading to 'worldcup-team-stats'
   82 docs → 1 batches


  ✅ Done — 82 docs in 'worldcup-team-stats'

📤 Uploading to 'worldcup-2026'
   47 docs → 1 batches


  ✅ Done — 47 docs in 'worldcup-2026'

📤 Uploading to 'worldcup-matches'
   964 docs → 10 batches


  ✅ Done — 964 docs in 'worldcup-matches'

📤 Uploading to 'international-matches'
   49,071 docs → 491 batches


  ✅ Done — 49,071 docs in 'international-matches'

✅ All 4 indexes populated in Pinecone


In [40]:
# Verify counts + test search on all 4 indexes

time.sleep(5)  # allow Pinecone to update counts

print("📊 Final Pinecone vector counts:\n")
print(f"{'Index':<30} {'Vectors':>10}")
print("─" * 42)
for label, name in INDEXES.items():
    count = pc.Index(name).describe_index_stats().get("total_vector_count", 0)
    status = "✅" if count > 0 else "❌"
    print(f"  {status} {name:<28} {count:>10,}")

# Test searches
def test_search(label, query):
    vs      = PineconeVectorStore(index_name=INDEXES[label], embedding=embeddings)
    results = vs.similarity_search(query, k=2)
    print(f"\n🔍 [{INDEXES[label]}]")
    print(f"   Query: '{query}'")
    for i, doc in enumerate(results, 1):
        print(f"   {i}. {doc.page_content[:120]}...")

print("\n🧪 Search tests:\n")
test_search("wc_matches",   "Germany Brazil 2014 semifinal goals scored")
test_search("intl_matches", "Argentina England Maradona 1986")
test_search("team_stats",   "Brazil all time top scorers World Cup")
test_search("wc_2026",      "Which teams qualified for 2026 World Cup")

print("\n✅ Data pipeline 100% complete!")
print("   All 4 Pinecone indexes ready")

📊 Final Pinecone vector counts:

Index                             Vectors
──────────────────────────────────────────
  ✅ worldcup-matches                    964
  ✅ international-matches            49,071
  ✅ worldcup-team-stats                  82
  ✅ worldcup-2026                        47

🧪 Search tests:


🔍 [worldcup-matches]
   Query: 'Germany Brazil 2014 semifinal goals scored'
   1. In the 2014 FIFA World Cup in Belo Horizonte, Germany defeated Brazil 7–1 in a dominant performance. It was a high-scori...
   2. In the 2014 FIFA World Cup in Salvador, Germany defeated Portugal 4–0 comfortably. Scorers — Germany: Thomas Müller (pen...

🔍 [international-matches]
   Query: 'Argentina England Maradona 1986'
   1. In the 1986 FIFA World Cup in Mexico City, Argentina defeated England 2–1 narrowly. Scorers — Argentina: Diego Maradona,...
   2. In the 1986 FIFA World Cup in Mexico City, Argentina defeated Belgium 2–0 narrowly. Scorers — Argentina: Diego Maradona,...

🔍 [worldcup-team-st

In [43]:
# Test all 4 Pinecone indexes with real queries

def search(label: str, query: str, k: int = 3):
    """Search a Pinecone index and show results."""
    vs      = PineconeVectorStore(
        index_name=INDEXES[label],
        embedding=embeddings
    )
    results = vs.similarity_search_with_score(query, k=k)

    print(f"\n{'─'*60}")
    print(f"🔍 Index:  {INDEXES[label]}")
    print(f"   Query:  '{query}'")
    print(f"{'─'*60}")

    if not results:
        print("   ❌ No results returned")
        return

    for i, (doc, score) in enumerate(results, 1):
        relevance = 1 - score
        print(f"\n   Result {i} (relevance={relevance:.3f}):")
        print(f"   {doc.page_content[:150]}...")
        print(f"   Metadata: {dict(list(doc.metadata.items())[:4])}")


# ── Test 1: Famous match with scorer data ─────────────────
search("wc_matches", "Germany Brazil 2014 semifinal Müller goals")

# ── Test 2: Penalty shootout match ────────────────────────
search("wc_matches", "penalty shootout World Cup final Argentina France 2022")

# ── Test 3: Team stats ────────────────────────────────────
search("team_stats", "Brazil World Cup wins appearances top scorers")

# ── Test 4: International H2H ─────────────────────────────
search("intl_matches", "Argentina England Maradona hand of god 1986")

# ── Test 5: 2026 preview ──────────────────────────────────
search("wc_2026", "Which teams qualified for 2026 World Cup")

# ── Test 6: Metadata filter ───────────────────────────────
print(f"\n{'─'*60}")
print("🔍 Testing metadata filter — World Cup Finals only")
print(f"{'─'*60}")

vs      = PineconeVectorStore(
    index_name=INDEXES["wc_matches"],
    embedding=embeddings
)
results = vs.similarity_search(
    "World Cup final winner",
    k=5,
    filter={"is_final": True}
)

print(f"\n   Found {len(results)} finals:")
for doc in results:
    print(f"   • {doc.page_content[:100]}...")

print(f"\n{'─'*60}")
print("✅ All search tests complete")



────────────────────────────────────────────────────────────
🔍 Index:  worldcup-matches
   Query:  'Germany Brazil 2014 semifinal Müller goals'
────────────────────────────────────────────────────────────

   Result 1 (relevance=0.359):
   In the 2014 FIFA World Cup in Salvador, Germany defeated Portugal 4–0 comfortably. Scorers — Germany: Thomas Müller (pen), Mats Hummels, Thomas Müller...
   Metadata: {'away_goals': 0.0, 'away_team': 'Portugal', 'city': 'Salvador', 'date': '2014-06-16'}

   Result 2 (relevance=0.366):
   In the 2014 FIFA World Cup in Belo Horizonte, Germany defeated Brazil 7–1 in a dominant performance. It was a high-scoring game with 8 total goals. Sc...
   Metadata: {'away_goals': 7.0, 'away_team': 'Germany', 'city': 'Belo Horizonte', 'date': '2014-07-08'}

   Result 3 (relevance=0.410):
   In the 2014 FIFA World Cup in Recife, Germany defeated United States 1–0 narrowly. Scorers — United States: none. Germany: Thomas Müller....
   Metadata: {'away_goals': 1.0, 'a

In [64]:

import json
import plotly.graph_objects as go
from pathlib import Path
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_pinecone import PineconeVectorStore

# ── Define all clients here so cell is fully self-contained ──
os.environ["OPENAI_API_KEY"]   = userdata.get("OPENAI_API_KEY")
os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm        = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
pc         = Pinecone(api_key=userdata.get("PINECONE_API_KEY"))


# Index names
INDEXES = {
    "wc_matches":   "worldcup-matches",
    "intl_matches": "international-matches",
    "team_stats":   "worldcup-team-stats",
    "wc_2026":      "worldcup-2026",
}

# Helpers
def get_vs(label: str) -> PineconeVectorStore:
    return PineconeVectorStore(
        index_name=INDEXES[label],
        embedding=embeddings
    )

PREFS_FILE = Path("/content/user_prefs.json")

def load_prefs() -> dict:
    if PREFS_FILE.exists():
        return json.loads(PREFS_FILE.read_text())
    return {
        "favorite_team":    "",
        "language":         "English",
        "detail_level":     "medium",
        "preferred_format": "paragraph",
    }

def save_pref(key: str, value: str) -> dict:
    prefs      = load_prefs()
    prefs[key] = value
    PREFS_FILE.write_text(json.dumps(prefs, indent=2))
    return prefs

In [65]:
# TOOL 1 — match_retrieval_tool

@tool
def match_retrieval_tool(query: str) -> str:
    """
    Searches FIFA World Cup match history from 1930 to 2022.
    Use for questions about specific match results, scores,
    goalscorers, penalty shootouts, or any historical World Cup fact.
    Do NOT use for team statistics or 2026 preview questions.
    Input: natural language question about a World Cup match.
    Returns: relevant match records with scorer details.
    """
    vs      = get_vs("wc_matches")
    results = vs.similarity_search_with_score(query, k=8)

    if not results:
        return "No relevant World Cup match records found."

    filtered = [(doc, score) for doc, score in results if score < 1.0]
    if not filtered:
        return "No closely matching records found. Try rephrasing with specific team names or years."

    return "\n\n".join(
        f"[relevance={round(1-score,3)}] {doc.page_content}"
        for doc, score in filtered
    )


In [66]:
# TOOL 2 — team_stats_tool

@tool
def team_stats_tool(query: str) -> str:
    """
    Retrieves FIFA World Cup statistics for one or more teams.
    Use for questions about win rates, tournament appearances,
    goals scored, best stage reached, all-time top scorers,
    or penalty shootout records in World Cups.
    Do NOT use for specific match results or 2026 questions.
    Input: question about team stats e.g. 'Brazil World Cup record'
    Returns: detailed team statistics.
    """
    vs      = get_vs("team_stats")
    results = vs.similarity_search_with_score(query, k=5)

    if not results:
        return "No team statistics found for that query."

    filtered = [(doc, score) for doc, score in results if score < 1.2]
    if not filtered:
        return "Could not find stats for that team. Use the full country name e.g. 'Germany' not 'GER'."

    return "\n\n".join(
        f"[relevance={round(1-score,3)}] {doc.page_content}"
        for doc, score in filtered
    )

In [67]:

# TOOL 3 — head_to_head_tool

@tool
def head_to_head_tool(query: str) -> str:
    """
    Computes head-to-head record and recent form for two teams
    across ALL international matches including friendlies.
    Use when comparing two teams or preparing a match prediction.
    Always call this BEFORE match_prediction_tool.
    Input format: 'Team A vs Team B' e.g. 'Brazil vs Germany'
    Returns: H2H win/draw/loss record and last 5 matches per team.
    """
    if "vs" not in query.lower():
        return "Input must be 'Team A vs Team B'"

    parts  = query.split("vs", 1)
    team_a = parts[0].strip().title()
    team_b = parts[1].strip().title()

 # H2H record
    h2h = df_all[
        ((df_all["home_team"].str.lower() == team_a.lower()) &
         (df_all["away_team"].str.lower() == team_b.lower())) |
        ((df_all["home_team"].str.lower() == team_b.lower()) &
         (df_all["away_team"].str.lower() == team_a.lower()))
    ].sort_values("date").copy()

    if h2h.empty:
        h2h_text = f"No international meetings found between {team_a} and {team_b}."
        a_wins = b_wins = draws = 0
    else:
        a_wins = sum(
            (r["home_team"].lower() == team_a.lower() and r["home_goals"] > r["away_goals"]) or
            (r["away_team"].lower() == team_a.lower() and r["away_goals"] > r["home_goals"])
            for _, r in h2h.iterrows()
        )
        b_wins = sum(
            (r["home_team"].lower() == team_b.lower() and r["home_goals"] > r["away_goals"]) or
            (r["away_team"].lower() == team_b.lower() and r["away_goals"] > r["home_goals"])
            for _, r in h2h.iterrows()
        )
        draws = len(h2h) - a_wins - b_wins

        recent_lines = "\n".join(
            f"  {str(r['date'].date())} ({r['tournament']}): "
            f"{r['home_team']} {int(r['home_goals'])}–{int(r['away_goals'])} {r['away_team']}"
            for _, r in h2h.tail(5).iterrows()
        )
        h2h_text = (
            f"Head-to-Head ({len(h2h)} meetings):\n"
            f"  {team_a}: {a_wins} wins | Draws: {draws} | {team_b}: {b_wins} wins\n\n"
            f"Last 5 meetings:\n{recent_lines}"
        )

# Recent form
    def recent_form(team: str) -> str:
        mask   = (
            (df_all["home_team"].str.lower() == team.lower()) |
            (df_all["away_team"].str.lower() == team.lower())
        )
        recent = df_all[mask].sort_values("date", ascending=False).head(5)
        if recent.empty:
            return f"  No data found for {team}"
        lines = []
        for _, r in recent.iterrows():
            is_home = r["home_team"].lower() == team.lower()
            hg, ag  = int(r["home_goals"]), int(r["away_goals"])
            outcome = (
                "W" if (is_home and hg > ag) or (not is_home and ag > hg)
                else "D" if hg == ag else "L"
            )
            lines.append(
                f"  {str(r['date'].date())} [{outcome}] "
                f"{r['home_team']} {hg}–{ag} {r['away_team']} ({r['tournament']})"
            )
        return "\n".join(lines)

    # Cache for chart in prediction tool
    head_to_head_tool._cache = {
        "team_a": team_a, "team_b": team_b,
        "a_wins": a_wins, "b_wins": b_wins,
        "draws":  draws,  "total":  len(h2h),
    }

    return (
        f"{h2h_text}\n\n"
        f"{team_a} Recent Form:\n{recent_form(team_a)}\n\n"
        f"{team_b} Recent Form:\n{recent_form(team_b)}"
    )

head_to_head_tool._cache = {}


In [68]:
# TOOL 4 — match_prediction_tool

PREDICTION_PROMPT = PromptTemplate(
    input_variables=["team_a","team_b","h2h_data",
                     "language","detail_level","preferred_format","favorite_team"],
    template="""You are a professional FIFA World Cup analyst.
Write a match preview in exactly 3 paragraphs:

Paragraph 1 — Head-to-head history and rivalry patterns
Paragraph 2 — Recent form and key strengths of each team
Paragraph 3 — Predicted winner, scoreline, and reasoning

User preferences:
  Language:      {language}
  Detail level:  {detail_level}
  Format:        {preferred_format}
  Favorite team: {favorite_team}

Teams: {team_a} vs {team_b}

Data:
{h2h_data}

Match Preview:"""
)

@tool
def match_prediction_tool(query: str) -> str:
    """
    Generates a full match preview and predicted outcome for two teams.
    Always call head_to_head_tool first before calling this tool.
    Input format: 'Team A vs Team B' e.g. 'Argentina vs France'
    Returns: 3-paragraph match preview with predicted winner and scoreline.
    Also displays a head-to-head bar chart.
    """
    if "vs" not in query.lower():
        return "Input must be 'Team A vs Team B'"

    parts  = query.split("vs", 1)
    team_a = parts[0].strip().title()
    team_b = parts[1].strip().title()

    # Get H2H data + populate cache for chart
    h2h_data = head_to_head_tool.invoke(f"{team_a} vs {team_b}")

    # Generate prediction
    prefs    = load_prefs()
    chain    = PREDICTION_PROMPT | llm
    response = chain.invoke({
        "team_a":          team_a,
        "team_b":          team_b,
        "h2h_data":        h2h_data,
        "language":        prefs["language"],
        "detail_level":    prefs["detail_level"],
        "preferred_format":prefs["preferred_format"],
        "favorite_team":   prefs["favorite_team"],
    })

    # Show chart
    _show_h2h_chart(team_a, team_b)

    return response.content


def _show_h2h_chart(team_a: str, team_b: str):
    c = head_to_head_tool._cache
    fig = go.Figure(go.Bar(
        x=[f"{team_a} Wins", "Draws", f"{team_b} Wins"],
        y=[c.get("a_wins",0), c.get("draws",0), c.get("b_wins",0)],
        marker_color=["#3b82f6","#94a3b8","#ef4444"],
        text=[c.get("a_wins",0), c.get("draws",0), c.get("b_wins",0)],
        textposition="auto",
        textfont=dict(size=14, color="white"),
    ))
    fig.update_layout(
        title=f"⚽ {team_a} vs {team_b} — All-time H2H ({c.get('total',0)} meetings)",
        template="plotly_dark",
        height=380,
        showlegend=False,
        yaxis_title="Matches",
    )
    fig.show()


In [69]:
# TOOL 5 — wc2026_tool

@tool
def wc2026_tool(query: str) -> str:
    """
    Retrieves information about the 2026 FIFA World Cup.
    Use for questions about qualified teams, host nations,
    host cities, the 48-team format, or any 2026 tournament details.
    Do NOT use for historical match results or team statistics.
    Input: question about the 2026 World Cup.
    Returns: relevant 2026 tournament information.
    """
    vs      = get_vs("wc_2026")
    results = vs.similarity_search_with_score(query, k=6)

    if not results:
        return "No 2026 data found. The 2026 FIFA World Cup is hosted by USA, Canada, and Mexico with 48 teams."

    filtered = [(doc, score) for doc, score in results if score < 1.2]
    if not filtered:
        return "Limited 2026 data available. Tournament hosted by USA, Canada, Mexico with 48 teams."

    return "\n\n".join(
        f"[relevance={round(1-score,3)}] {doc.page_content}"
        for doc, score in filtered
    )


In [70]:
# MEMORY TOOLS — Preference management

@tool
def set_preference_tool(query: str) -> str:
    """
    Saves a user preference that persists across sessions.
    Input format: 'key=value'
    Valid keys: favorite_team, language, detail_level, preferred_format
    Example: 'favorite_team=Brazil' or 'language=Spanish'
    """
    try:
        key, value = query.split("=", 1)
        key        = key.strip().lower()
        value      = value.strip()
        valid      = ["favorite_team","language","detail_level","preferred_format"]
        if key not in valid:
            return f"Invalid key '{key}'. Valid: {', '.join(valid)}"
        prefs = save_pref(key, value)
        return f"✅ Saved: {key} = {value}\nAll prefs: {json.dumps(prefs, indent=2)}"
    except ValueError:
        return "Format: 'key=value' e.g. 'favorite_team=Brazil'"


@tool
def get_preferences_tool(query: str) -> str:
    """Returns all saved user preferences."""
    prefs = load_prefs()
    return "Current preferences:\n" + "\n".join(f"  {k}: {v}" for k, v in prefs.items())

In [71]:
# REGISTER ALL TOOLS

ALL_TOOLS = [
    match_retrieval_tool,
    team_stats_tool,
    head_to_head_tool,
    match_prediction_tool,
    wc2026_tool,
    set_preference_tool,
    get_preferences_tool,
]

print("✅ All tools defined\n")
print(f"  {'#':<4} {'Tool':<30} {'Stage'}")
print("  " + "─" * 55)
rows = [
    ("1", "match_retrieval_tool",  "Retrieval — WC history"),
    ("2", "team_stats_tool",       "Retrieval — Team stats"),
    ("3", "head_to_head_tool",     "Reasoning — H2H + Form"),
    ("4", "match_prediction_tool", "Report — Prediction + Chart"),
    ("5", "wc2026_tool",           "Retrieval — 2026 preview"),
    ("M", "set_preference_tool",   "Memory — Save"),
    ("M", "get_preferences_tool",  "Memory — Read"),
]
for num, name, stage in rows:
    print(f"  {num:<4} {name:<30} {stage}")

print(f"\n  Total: {len(ALL_TOOLS)} tools")


✅ All tools defined

  #    Tool                           Stage
  ───────────────────────────────────────────────────────
  1    match_retrieval_tool           Retrieval — WC history
  2    team_stats_tool                Retrieval — Team stats
  3    head_to_head_tool              Reasoning — H2H + Form
  4    match_prediction_tool          Report — Prediction + Chart
  5    wc2026_tool                    Retrieval — 2026 preview
  M    set_preference_tool            Memory — Save
  M    get_preferences_tool           Memory — Read

  Total: 7 tools


In [72]:
# QUICK TESTS — Run all 5 tools

print("\n" + "="*55)
print("🧪 Running quick tests...\n")

# Test 1
print("Test 1 — match_retrieval_tool")
r1 = match_retrieval_tool.invoke("Germany Brazil 2014 semifinal goals")
print(f"  {r1[:120]}...\n")

# Test 2
print("Test 2 — team_stats_tool")
r2 = team_stats_tool.invoke("Brazil World Cup wins appearances")
print(f"  {r2[:120]}...\n")

# Test 3
print("Test 3 — head_to_head_tool")
r3 = head_to_head_tool.invoke("Brazil vs Germany")
print(f"  {r3[:120]}...\n")

# Test 4
print("Test 4 — match_prediction_tool (generates chart)")
r4 = match_prediction_tool.invoke("Brazil vs Germany")
print(f"  {r4[:120]}...\n")

# Test 5
print("Test 5 — wc2026_tool")
r5 = wc2026_tool.invoke("Which teams qualified for 2026 World Cup")
print(f"  {r5[:120]}...\n")

# Test 6
print("Test 6 — preference tools")
set_preference_tool.invoke("favorite_team=Brazil")
r6 = get_preferences_tool.invoke("show")
print(f"  {r6}\n")

print("="*55)
print("✅ All tools tested — ready to build agent")


🧪 Running quick tests...

Test 1 — match_retrieval_tool
  [relevance=0.385] In the 2014 FIFA World Cup in Belo Horizonte, Germany defeated Brazil 7–1 in a dominant performance. I...

Test 2 — team_stats_tool
  [relevance=0.311] Brazil have participated in 22 FIFA World Cups ([1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970,...

Test 3 — head_to_head_tool
  Head-to-Head (23 meetings):
  Brazil: 13 wins | Draws: 5 | Germany: 5 wins

Last 5 meetings:
  2004-09-08 (Friendly): Ge...

Test 4 — match_prediction_tool (generates chart)


  The storied rivalry between Brazil and Germany is one of the most captivating in football history, characterized by inte...

Test 5 — wc2026_tool
  [relevance=0.281] United States have qualified for the 2026 FIFA World Cup, hosted jointly by the United States, Canada,...

Test 6 — preference tools
  Current preferences:
  favorite_team: Brazil
  language: English
  detail_level: medium
  preferred_format: paragraph

✅ All tools tested — ready to build agent


In [77]:
# Install LangGraph (required for new agent API)

!pip install -q langgraph langchain-core

In [78]:
# Build Agent using LangGraph

import os
import json
from pathlib import Path
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# LLM
agent_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# System prompt
SYSTEM_PROMPT = """You are an expert FIFA World Cup analyst chatbot.
You have deep knowledge of World Cup history from 1930 to 2022,
team statistics, match results, and the 2026 tournament preview.

You have access to these tools:

1. match_retrieval_tool
   → Search historical World Cup match results and goalscorers
   → Use for: "Who won X", "What was the score of Y", "Who scored in Z"

2. team_stats_tool
   → Get team statistics, win rates, appearances, top scorers
   → Use for: "How many times did X win", "What is X's record"

3. head_to_head_tool
   → Compute H2H record and recent form between two teams
   → Always call this BEFORE match_prediction_tool

4. match_prediction_tool
   → Generate match preview and predicted outcome with chart
   → Use for: "predict X vs Y"

5. wc2026_tool
   → Get 2026 World Cup qualified teams and tournament info
   → Use for: any question about the 2026 tournament

6. set_preference_tool
   → Save user preferences (language, favorite team, detail level)

7. get_preferences_tool
   → Show current saved preferences

Decision rules:
- History questions   → match_retrieval_tool
- Stats questions     → team_stats_tool
- Prediction requests → head_to_head_tool THEN match_prediction_tool
- 2026 questions      → wc2026_tool
- Always be concise, factual, and engaging
- If data is missing  → say you don't have that information"""

# Memory
memory   = MemorySaver()

# Build LangGraph React agent
agent = create_react_agent(
    model=agent_llm,
    tools=ALL_TOOLS,
    prompt=SYSTEM_PROMPT,
    checkpointer=memory,
)

print("✅ Agent built with LangGraph\n")
print(f"  LLM:      {agent_llm.model_name}")
print(f"  Tools:    {len(ALL_TOOLS)}")
print(f"  Memory:   LangGraph MemorySaver")

#  Config — thread_id keeps conversation separate
# Each user session gets its own thread_id
THREAD_CONFIG = {"configurable": {"thread_id": "worldcup_session_1"}}

# chat() function — used by UI
def chat(user_input: str, verbose: bool = False) -> str:
    """
    Send a message to the agent and get a response.
    Memory is maintained automatically via thread_id.
    """
    result = agent.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=THREAD_CONFIG,
    )

    # Extract final answer from messages
    messages = result["messages"]
    answer   = messages[-1].content

    if verbose:
        # Show tool calls made
        for msg in messages:
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  🔧 Tool called: {tc['name']}")

    return answer


def new_session():
    """Start a fresh conversation — clears memory."""
    global THREAD_CONFIG
    import time
    THREAD_CONFIG = {"configurable": {"thread_id": f"session_{int(time.time())}"}}
    print("✅ New session started")



✅ Agent built with LangGraph

  LLM:      gpt-4o-mini
  Tools:    7
  Memory:   LangGraph MemorySaver


In [79]:
# Test the agent with 4 question types

print("\n" + "="*55)
print("🧪 Testing agent...\n")

# Test 1 — History
print("Test 1: History question")
r1 = chat("Who scored in the 2014 World Cup semifinal between Germany and Brazil?", verbose=True)
print(f"🤖 {r1[:200]}\n")

# Test 2 — Stats
print("─"*40)
print("Test 2: Team stats")
r2 = chat("How many World Cups has Brazil won?", verbose=True)
print(f"🤖 {r2[:200]}\n")

# Test 3 — Prediction + chart
print("─"*40)
print("Test 3: Match prediction")
r3 = chat("Predict Argentina vs France", verbose=True)
print(f"🤖 {r3[:200]}\n")

# Test 4 — Follow-up (tests memory)
print("─"*40)
print("Test 4: Follow-up (tests memory)")
r4 = chat("What was the result of their last meeting?", verbose=True)
print(f"🤖 {r4[:200]}\n")

print("="*55)
print("✅ Agent tests complete — ready to build chat UI")


🧪 Testing agent...

Test 1: History question
  🔧 Tool called: match_retrieval_tool
🤖 In the 2014 World Cup semifinal between Germany and Brazil, the goalscorers were:

- **Brazil**: Oscar
- **Germany**: Thomas Müller, Miroslav Klose, Toni Kroos (twice), Sami Khedira, André Schürrle (t

────────────────────────────────────────
Test 2: Team stats
  🔧 Tool called: match_retrieval_tool
  🔧 Tool called: team_stats_tool
🤖 Brazil has won the FIFA World Cup a record **5 times**. The years in which they claimed the title are 1958, 1962, 1970, 1994, and 2002.

────────────────────────────────────────
Test 3: Match prediction


  🔧 Tool called: match_retrieval_tool
  🔧 Tool called: team_stats_tool
  🔧 Tool called: head_to_head_tool
  🔧 Tool called: match_prediction_tool
🤖 The upcoming match between Argentina and France is set to be an exciting clash, given their rich history and competitive rivalry. Argentina has a slight edge in their overall head-to-head record with 

────────────────────────────────────────
Test 4: Follow-up (tests memory)
  🔧 Tool called: match_retrieval_tool
  🔧 Tool called: team_stats_tool
  🔧 Tool called: head_to_head_tool
  🔧 Tool called: match_prediction_tool
  🔧 Tool called: match_retrieval_tool
🤖 The last meeting between Argentina and France took place in the **2022 FIFA World Cup final**, where the match ended in a dramatic **3-3 draw**. Argentina initially led the match with goals from Lione

✅ Agent tests complete — ready to build chat UI


In [ ]:
# CHAT UI — ipywidgets

import ipywidgets as widgets
from IPython.display import display, clear_output
import time

# Preference sidebar
fav_team   = widgets.Text(
    placeholder="e.g. Brazil",
    description="⭐ Fav Team:",
    layout=widgets.Layout(width="280px")
)
language   = widgets.Dropdown(
    options=["English","Spanish","French","Portuguese","German"],
    description="🌐 Language:",
    layout=widgets.Layout(width="280px")
)
detail     = widgets.Dropdown(
    options=["brief","medium","detailed"],
    value="medium",
    description="📊 Detail:",
    layout=widgets.Layout(width="280px")
)
fmt        = widgets.Dropdown(
    options=["paragraph","bullet points"],
    description="📝 Format:",
    layout=widgets.Layout(width="280px")
)
save_btn   = widgets.Button(
    description="💾 Save Preferences",
    button_style="info",
    layout=widgets.Layout(width="280px")
)
new_btn    = widgets.Button(
    description="🔄 New Session",
    button_style="warning",
    layout=widgets.Layout(width="280px")
)
prefs_out  = widgets.Output()

# Load saved prefs into widgets
def load_prefs_to_widgets():
    prefs = load_prefs()
    fav_team.value = prefs.get("favorite_team", "")
    language.value = prefs.get("language", "English")
    detail.value   = prefs.get("detail_level", "medium")
    fmt.value      = prefs.get("preferred_format", "paragraph")

load_prefs_to_widgets()

def on_save_prefs(b):
    with prefs_out:
        clear_output()
        save_pref("favorite_team",    fav_team.value)
        save_pref("language",         language.value)
        save_pref("detail_level",     detail.value)
        save_pref("preferred_format", fmt.value)
        print("✅ Preferences saved!")

def on_new_session(b):
    with chat_out:
        clear_output()
        _display_message("🤖 Bot", "New session started! Ask me anything about the World Cup.")
    with prefs_out:
        clear_output()
        print("✅ Session cleared")
    new_session()

save_btn.on_click(on_save_prefs)
new_btn.on_click(on_new_session)

sidebar = widgets.VBox([
    widgets.HTML("<h3 style='color:#3b82f6'>⚙️ Preferences</h3>"),
    fav_team, language, detail, fmt,
    save_btn, new_btn,
    widgets.HTML("<hr>"),
    widgets.HTML("""<small>
    <b>💡 Try asking:</b><br>
    • Who won the 2014 final?<br>
    • Brazil's World Cup record<br>
    • Predict Brazil vs Germany<br>
    • Who qualified for 2026?<br>
    • Who scored the most WC goals?
    </small>"""),
    prefs_out,
], layout=widgets.Layout(
    width="300px",
    padding="10px",
    border="1px solid #374151",
    border_radius="8px",
))

# Chat area
chat_out   = widgets.Output(
    layout=widgets.Layout(
        height="450px",
        overflow_y="scroll",
        border="1px solid #374151",
        padding="10px",
        border_radius="8px",
    )
)
text_input = widgets.Text(
    placeholder="Ask about World Cup history or type: predict Brazil vs Germany",
    layout=widgets.Layout(width="75%", height="38px")
)
send_btn   = widgets.Button(
    description="Send ⚽",
    button_style="primary",
    layout=widgets.Layout(width="12%", height="38px")
)
clear_btn  = widgets.Button(
    description="Clear",
    button_style="warning",
    layout=widgets.Layout(width="10%", height="38px")
)
status_lbl = widgets.Label(value="")

def _display_message(sender: str, message: str):
    """Display a chat message in the output area."""
    color = "#3b82f6" if "Bot" in sender else "#10b981"
    with chat_out:
        display(widgets.HTML(
            f"<div style='margin:6px 0'>"
            f"<b style='color:{color}'>{sender}:</b> "
            f"<span style='white-space:pre-wrap'>{message}</span>"
            f"</div>"
        ))

def on_send(b):
    user_msg = text_input.value.strip()
    if not user_msg:
        return

    text_input.value   = ""
    status_lbl.value   = "⏳ Thinking..."

    # Show user message
    _display_message("🧑 You", user_msg)

    try:
        # Get agent response
        response = chat(user_msg)
        _display_message("🤖 Bot", response)
    except Exception as e:
        _display_message("⚠️ Error", str(e))

    status_lbl.value = ""

def on_clear(b):
    with chat_out:
        clear_output()
    _display_message("🤖 Bot", "Chat cleared! Ask me anything about the World Cup.")

send_btn.on_click(on_send)
clear_btn.on_click(on_clear)
text_input.on_submit(on_send)   # Enter key sends message

# Initial welcome message
_display_message(
    "🤖 Bot",
    "👋 Welcome to the FIFA World Cup Analyst Chatbot!\n\n"
    "I can answer questions about:\n"
    "• World Cup history (1930–2022)\n"
    "• Team statistics and records\n"
    "• Match predictions with charts\n"
    "• 2026 World Cup preview\n\n"
    "Try: 'Who won the 2018 World Cup?' or 'predict Brazil vs Germany'"
)

# Assemble full UI
header = widgets.HTML(
    "<h2 style='color:#f59e0b;margin:0'>🏆 FIFA World Cup Analyst Chatbot</h2>"
    "<p style='color:#9ca3af;margin:4px 0'>Powered by LangChain · Pinecone · OpenAI</p>"
    "<hr style='border-color:#374151'>"
)
input_row  = widgets.HBox([text_input, send_btn, clear_btn, status_lbl])
chat_panel = widgets.VBox([chat_out, input_row])
main_panel = widgets.HBox(
    [chat_panel, sidebar],
    layout=widgets.Layout(gap="12px")
)

display(widgets.VBox([header, main_panel]))

In [82]:
# Write app.py to Colab disk

app_code = '''
import os
import json
import sys
import streamlit as st
import plotly.graph_objects as go
from pathlib import Path
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from pinecone import Pinecone
import pandas as pd

# ── Page config ───────────────────────────────────────────
st.set_page_config(
    page_title="FIFA World Cup Chatbot",
    page_icon="🏆",
    layout="wide"
)

# ── API Keys ──────────────────────────────────────────────
OPENAI_API_KEY   = os.environ.get("OPENAI_API_KEY", "")
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY", "")

os.environ["OPENAI_API_KEY"]   = OPENAI_API_KEY
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

# ── Indexes ───────────────────────────────────────────────
INDEXES = {
    "wc_matches":   "worldcup-matches",
    "intl_matches": "international-matches",
    "team_stats":   "worldcup-team-stats",
    "wc_2026":      "worldcup-2026",
}

# ── Cached resources — only init once ─────────────────────
@st.cache_resource
def init_resources():
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    llm        = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
    pc         = Pinecone(api_key=PINECONE_API_KEY)
    return embeddings, llm, pc

embeddings, llm, pc = init_resources()

def get_vs(label):
    return PineconeVectorStore(
        index_name=INDEXES[label],
        embedding=embeddings
    )

# ── User preferences ──────────────────────────────────────
PREFS_FILE = Path("/content/user_prefs.json")

def load_prefs():
    if PREFS_FILE.exists():
        return json.loads(PREFS_FILE.read_text())
    return {
        "favorite_team": "",
        "language": "English",
        "detail_level": "medium",
        "preferred_format": "paragraph",
    }

def save_pref(key, value):
    prefs      = load_prefs()
    prefs[key] = value
    PREFS_FILE.write_text(json.dumps(prefs, indent=2))

# ── Tools ─────────────────────────────────────────────────
@tool
def match_retrieval_tool(query: str) -> str:
    """
    Searches FIFA World Cup match history from 1930 to 2022.
    Use for questions about specific match results, scores,
    goalscorers, penalty shootouts, or any historical World Cup fact.
    Do NOT use for team statistics or 2026 preview questions.
    Input: natural language question about a World Cup match.
    """
    vs      = get_vs("wc_matches")
    results = vs.similarity_search_with_score(query, k=8)
    if not results:
        return "No relevant World Cup match records found."
    filtered = [(doc, score) for doc, score in results if score < 1.0]
    if not filtered:
        return "No closely matching records found."
    return "\\n\\n".join(
        f"[relevance={round(1-score,3)}] {doc.page_content}"
        for doc, score in filtered
    )


@tool
def team_stats_tool(query: str) -> str:
    """
    Retrieves FIFA World Cup statistics for one or more teams.
    Use for questions about win rates, tournament appearances,
    goals scored, best stage reached, all-time top scorers,
    or penalty shootout records in World Cups.
    Input: question about team stats e.g. Brazil World Cup record.
    """
    vs      = get_vs("team_stats")
    results = vs.similarity_search_with_score(query, k=5)
    if not results:
        return "No team statistics found."
    filtered = [(doc, score) for doc, score in results if score < 1.2]
    if not filtered:
        return "Could not find stats. Use full country name e.g. Germany not GER."
    return "\\n\\n".join(
        f"[relevance={round(1-score,3)}] {doc.page_content}"
        for doc, score in filtered
    )


@tool
def head_to_head_tool(query: str) -> str:
    """
    Computes head-to-head record and recent form for two teams.
    Always call this BEFORE match_prediction_tool.
    Input format: Team A vs Team B e.g. Brazil vs Germany.
    """
    if "vs" not in query.lower():
        return "Input must be Team A vs Team B"

    parts  = query.split("vs", 1)
    team_a = parts[0].strip().title()
    team_b = parts[1].strip().title()

    # Load df from session state
    df = st.session_state.get("df_all")
    if df is None:
        return "Match data not available."

    h2h = df[
        ((df["home_team"].str.lower() == team_a.lower()) &
         (df["away_team"].str.lower() == team_b.lower())) |
        ((df["home_team"].str.lower() == team_b.lower()) &
         (df["away_team"].str.lower() == team_a.lower()))
    ].sort_values("date").copy()

    if h2h.empty:
        a_wins = b_wins = draws = 0
        h2h_text = f"No international meetings found between {team_a} and {team_b}."
    else:
        a_wins = sum(
            (r["home_team"].lower() == team_a.lower() and r["home_goals"] > r["away_goals"]) or
            (r["away_team"].lower() == team_a.lower() and r["away_goals"] > r["home_goals"])
            for _, r in h2h.iterrows()
        )
        b_wins = sum(
            (r["home_team"].lower() == team_b.lower() and r["home_goals"] > r["away_goals"]) or
            (r["away_team"].lower() == team_b.lower() and r["away_goals"] > r["home_goals"])
            for _, r in h2h.iterrows()
        )
        draws = len(h2h) - a_wins - b_wins
        recent = "\\n".join(
            f"  {str(r[\'date\'].date())} ({r[\'tournament\']}): "
            f"{r[\'home_team\']} {int(r[\'home_goals\'])}–{int(r[\'away_goals\'])} {r[\'away_team\']}"
            for _, r in h2h.tail(5).iterrows()
        )
        h2h_text = (
            f"H2H ({len(h2h)} meetings): "
            f"{team_a} {a_wins}W | {draws}D | {b_wins}W {team_b}\\n"
            f"Last 5:\\n{recent}"
        )

    # Store for chart
    st.session_state["h2h_cache"] = {
        "team_a": team_a, "team_b": team_b,
        "a_wins": a_wins, "b_wins": b_wins,
        "draws": draws,   "total": len(h2h),
    }

    def form(team):
        mask   = (
            (df["home_team"].str.lower() == team.lower()) |
            (df["away_team"].str.lower() == team.lower())
        )
        recent = df[mask].sort_values("date", ascending=False).head(5)
        if recent.empty:
            return f"No data for {team}"
        lines = []
        for _, r in recent.iterrows():
            is_home = r["home_team"].lower() == team.lower()
            hg, ag  = int(r["home_goals"]), int(r["away_goals"])
            outcome = (
                "W" if (is_home and hg > ag) or (not is_home and ag > hg)
                else "D" if hg == ag else "L"
            )
            lines.append(f"  {str(r[\'date\'].date())} [{outcome}] {r[\'home_team\']} {hg}–{ag} {r[\'away_team\']}")
        return "\\n".join(lines)

    return (
        f"{h2h_text}\\n\\n"
        f"{team_a} Recent Form:\\n{form(team_a)}\\n\\n"
        f"{team_b} Recent Form:\\n{form(team_b)}"
    )


PREDICTION_PROMPT = PromptTemplate(
    input_variables=["team_a","team_b","h2h_data",
                     "language","detail_level","preferred_format","favorite_team"],
    template="""You are a professional FIFA World Cup analyst.
Write a match preview in exactly 3 paragraphs:
Paragraph 1 — Head-to-head history and rivalry patterns
Paragraph 2 — Recent form and key strengths of each team
Paragraph 3 — Predicted winner, scoreline, and reasoning

Language: {language} | Detail: {detail_level} | Format: {preferred_format}
Favorite team: {favorite_team}

Teams: {team_a} vs {team_b}
Data: {h2h_data}

Match Preview:"""
)

@tool
def match_prediction_tool(query: str) -> str:
    """
    Generates a full match preview and prediction for two teams.
    Always call head_to_head_tool first.
    Input format: Team A vs Team B e.g. Argentina vs France.
    """
    if "vs" not in query.lower():
        return "Input must be Team A vs Team B"

    parts  = query.split("vs", 1)
    team_a = parts[0].strip().title()
    team_b = parts[1].strip().title()

    h2h_data = head_to_head_tool.invoke(f"{team_a} vs {team_b}")
    prefs    = load_prefs()
    chain    = PREDICTION_PROMPT | llm
    response = chain.invoke({
        "team_a": team_a, "team_b": team_b,
        "h2h_data": h2h_data,
        "language": prefs["language"],
        "detail_level": prefs["detail_level"],
        "preferred_format": prefs["preferred_format"],
        "favorite_team": prefs["favorite_team"],
    })

    # Flag chart to render after response
    st.session_state["show_chart"] = True
    return response.content


@tool
def wc2026_tool(query: str) -> str:
    """
    Retrieves information about the 2026 FIFA World Cup.
    Use for questions about qualified teams, host nations,
    host cities, or any 2026 tournament details.
    Input: question about the 2026 World Cup.
    """
    vs      = get_vs("wc_2026")
    results = vs.similarity_search_with_score(query, k=6)
    if not results:
        return "2026 World Cup hosted by USA, Canada, Mexico with 48 teams."
    filtered = [(doc, score) for doc, score in results if score < 1.2]
    if not filtered:
        return "Limited 2026 data. Tournament hosted by USA, Canada, Mexico."
    return "\\n\\n".join(
        f"[relevance={round(1-score,3)}] {doc.page_content}"
        for doc, score in filtered
    )


@tool
def set_preference_tool(query: str) -> str:
    """
    Saves a user preference. Input format: key=value
    Valid keys: favorite_team, language, detail_level, preferred_format
    """
    try:
        key, value = query.split("=", 1)
        key = key.strip().lower()
        valid = ["favorite_team","language","detail_level","preferred_format"]
        if key not in valid:
            return f"Invalid key. Valid: {\', \'.join(valid)}"
        save_pref(key, value.strip())
        return f"Saved: {key} = {value.strip()}"
    except ValueError:
        return "Format: key=value"


@tool
def get_preferences_tool(query: str) -> str:
    """Returns all saved user preferences."""
    prefs = load_prefs()
    return "\\n".join(f"{k}: {v}" for k, v in prefs.items())


ALL_TOOLS = [
    match_retrieval_tool, team_stats_tool,
    head_to_head_tool, match_prediction_tool,
    wc2026_tool, set_preference_tool, get_preferences_tool,
]

# ── Agent ─────────────────────────────────────────────────
SYSTEM_PROMPT = """You are an expert FIFA World Cup analyst.
Use match_retrieval_tool for history questions.
Use team_stats_tool for team record questions.
Use head_to_head_tool THEN match_prediction_tool for predictions.
Use wc2026_tool for 2026 questions.
Be concise and factual."""

@st.cache_resource
def build_agent():
    memory = MemorySaver()
    return create_react_agent(
        model=llm,
        tools=ALL_TOOLS,
        prompt=SYSTEM_PROMPT,
        checkpointer=memory,
    )

agent = build_agent()

def chat(user_input: str, thread_id: str) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = agent.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=config,
    )
    return result["messages"][-1].content

# ── H2H chart ─────────────────────────────────────────────
def show_h2h_chart():
    c = st.session_state.get("h2h_cache", {})
    if not c:
        return
    team_a = c.get("team_a", "Team A")
    team_b = c.get("team_b", "Team B")
    fig = go.Figure(go.Bar(
        x=[f"{team_a} Wins", "Draws", f"{team_b} Wins"],
        y=[c.get("a_wins",0), c.get("draws",0), c.get("b_wins",0)],
        marker_color=["#3b82f6","#94a3b8","#ef4444"],
        text=[c.get("a_wins",0), c.get("draws",0), c.get("b_wins",0)],
        textposition="auto",
        textfont=dict(size=14, color="white"),
    ))
    fig.update_layout(
        title=f"⚽ {team_a} vs {team_b} — All-time H2H ({c.get(\'total\',0)} meetings)",
        template="plotly_dark", height=350,
        showlegend=False, yaxis_title="Matches",
    )
    st.plotly_chart(fig, use_container_width=True)

# ══════════════════════════════════════════════════════════
# STREAMLIT UI
# ══════════════════════════════════════════════════════════

# Session state init
if "messages"   not in st.session_state: st.session_state.messages   = []
if "thread_id"  not in st.session_state: st.session_state.thread_id  = "session_1"
if "show_chart" not in st.session_state: st.session_state.show_chart = False
if "h2h_cache"  not in st.session_state: st.session_state.h2h_cache  = {}
if "df_all"     not in st.session_state: st.session_state.df_all     = None

# ── Header ────────────────────────────────────────────────
st.title("🏆 FIFA World Cup Analyst Chatbot")
st.caption("Powered by LangChain · LangGraph · Pinecone · OpenAI")

# ── Sidebar ───────────────────────────────────────────────
with st.sidebar:
    st.header("⚙️ Preferences")
    st.caption("Saved across sessions")

    prefs    = load_prefs()
    fav      = st.text_input("⭐ Favorite Team", value=prefs.get("favorite_team",""))
    lang     = st.selectbox("🌐 Language", ["English","Spanish","French","Portuguese","German"],
                             index=["English","Spanish","French","Portuguese","German"].index(prefs.get("language","English")))
    detail   = st.selectbox("📊 Detail Level", ["brief","medium","detailed"],
                             index=["brief","medium","detailed"].index(prefs.get("detail_level","medium")))
    fmt      = st.selectbox("📝 Format", ["paragraph","bullet points"],
                             index=["paragraph","bullet points"].index(prefs.get("preferred_format","paragraph")))

    if st.button("💾 Save Preferences"):
        save_pref("favorite_team",    fav)
        save_pref("language",         lang)
        save_pref("detail_level",     detail)
        save_pref("preferred_format", fmt)
        st.success("✅ Saved!")

    st.divider()

    if st.button("🔄 New Session"):
        import time
        st.session_state.thread_id  = f"session_{int(time.time())}"
        st.session_state.messages   = []
        st.session_state.show_chart = False
        st.rerun()

    st.divider()
    st.markdown("""
**💡 Try asking:**
- Who won the 2014 final?
- Brazil's World Cup record
- Predict Brazil vs Germany
- Who qualified for 2026?
- Who scored the most WC goals?
- Argentina vs France last meeting
""")

# ── Chat history ──────────────────────────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# Show H2H chart if prediction was made
if st.session_state.show_chart:
    show_h2h_chart()
    st.session_state.show_chart = False

# ── Chat input ────────────────────────────────────────────
if prompt := st.chat_input("Ask about World Cup history or type: predict Brazil vs Germany"):
    # Show user message
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Get agent response
    with st.chat_message("assistant"):
        with st.spinner("Analyzing..."):
            response = chat(prompt, st.session_state.thread_id)
            st.markdown(response)

            # Show chart if prediction was made
            if st.session_state.show_chart:
                show_h2h_chart()
                st.session_state.show_chart = False

    st.session_state.messages.append({"role": "assistant", "content": response})
'''

# Write app.py to disk
with open("/content/app.py", "w") as f:
    f.write(app_code)

print("✅ app.py written to /content/app.py")



✅ app.py written to /content/app.py
